## Text Representation

In [1]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
from sklearn.model_selection import GridSearchCV

from imblearn.combine import SMOTETomek


In [2]:
# Load dataset hasil preprocessing (No 3)
df = pd.read_csv("hsr_reviews_clean_preprocessed_final.csv")

# Pastikan tokens_clean berupa list, bukan string
if isinstance(df["tokens_clean"].iloc[0], str):
    df["tokens_clean"] = df["tokens_clean"].apply(ast.literal_eval)

print(df.info())
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   review_text   1000 non-null   object
 1   rating        1000 non-null   int64 
 2   len_char      1000 non-null   int64 
 3   len_word      1000 non-null   int64 
 4   tokens_clean  1000 non-null   object
 5   clean_text    999 non-null    object
dtypes: int64(3), object(3)
memory usage: 47.0+ KB
None


,review_text,rating,len_char,len_word,tokens_clean,clean_text
0,"I love this game, it's really F2P friendly, an...",5,493,89,"[love, game, really, free_to_play, friendly, n...",love game really free_to_play friendly not pus...
1,Child Phainon does not look like Phainon,2,40,7,"[child, phainon, not, look, like, phainon]",child phainon not look like phainon
2,Trash 🗑️🚮 you guys don't care about stelle mai...,1,117,21,"[trash, guy, not, care, stelle, main, anymore,...",trash guy not care stelle main anymore delete ...
3,maybe if this game had a better writer.....,1,43,8,"[maybe, game, well, writer]",maybe game well writer
4,50/50 is a complete lie its more like 75/25 on...,1,286,54,"[50, 50, complete, lie, like, 75, 25, banner, ...",50 50 complete lie like 75 25 banner iv notice...


In [3]:
df[df["clean_text"].isnull()]


,review_text,rating,len_char,len_word,tokens_clean,clean_text
692,just why?!,1,10,2,[],NaN


Terdapat 3 review yang setelah preprocessing menjadi teks kosong (hanya emoji / kata sangat umum yang dibuang sebagai stopword). Karena tidak mengandung informasi teks sama sekali, baris tersebut dihapus agar tidak menimbulkan masalah pada tahap text representation.

In [4]:
# Drop baris yang clean_text-nya NaN atau kosong string
mask_empty = df["clean_text"].isnull() | (df["clean_text"].astype(str).str.strip() == "")
df = df[~mask_empty].copy()

print("Jumlah baris setelah drop:", len(df))
print("Sisa NaN di clean_text:", df["clean_text"].isnull().sum())

Jumlah baris setelah drop: 999
Sisa NaN di clean_text: 0


In [5]:
# Fitur teks utama
X_text = df["clean_text"]
# Label rating
y = df["rating"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train_text))
print("Test size :", len(X_test_text))

print("\nDistribusi rating train:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nDistribusi rating test:")
print(y_test.value_counts(normalize=True).sort_index())

Train size: 799
Test size : 200

Distribusi rating train:
rating
1    0.531915
2    0.081352
3    0.060075
4    0.061327
5    0.265332
Name: proportion, dtype: float64

Distribusi rating test:
rating
1    0.535
2    0.080
3    0.060
4    0.060
5    0.265
Name: proportion, dtype: float64


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF: bobot pentingnya kata/bigram di setiap review
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigram + bigram
    min_df=5,             # hanya kata/bigram yang muncul di ≥ 5 dokumen
    max_df=0.9            # buang kata yang muncul di > 90% dokumen
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test  shape:", X_test_tfidf.shape)


TF-IDF train shape: (799, 661)
TF-IDF test  shape: (200, 661)


In [7]:
from gensim.models import Word2Vec
import numpy as np
import ast

# Pastikan tokens_clean berbentuk list, bukan string
if isinstance(df["tokens_clean"].iloc[0], str):
    df["tokens_clean"] = df["tokens_clean"].apply(ast.literal_eval)

# Ambil index train & test supaya sejajar dengan df
train_idx = X_train_text.index
test_idx  = X_test_text.index

# Kumpulan kalimat (list token) untuk Word2Vec
sentences = df.loc[train_idx.union(test_idx), "tokens_clean"].tolist()

w2v_dim = 100  # dimensi embedding

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=w2v_dim,
    window=5,
    min_count=3,
    sg=1,        # skip-gram
    workers=4,
    epochs=20,
    seed=42
)

print("Vocab size :", len(w2v_model.wv))
print("Embed dim  :", w2v_model.vector_size)


Vocab size : 948
Embed dim  : 100


In [8]:
def review_to_vec(tokens, model, dim):
    """
    Ubah list token menjadi satu vektor dokumen
    dengan merata-ratakan embedding semua kata yg ada di vocab.
    Jika semua token OOV → vektor nol.
    """
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

# Embedding untuk train
X_train_embed = np.vstack([
    review_to_vec(df.loc[i, "tokens_clean"], w2v_model, w2v_dim)
    for i in train_idx
])

# Embedding untuk test
X_test_embed = np.vstack([
    review_to_vec(df.loc[i, "tokens_clean"], w2v_model, w2v_dim)
    for i in test_idx
])

print("Embedding train shape:", X_train_embed.shape)
print("Embedding test  shape:", X_test_embed.shape)


Embedding train shape: (799, 100)
Embedding test  shape: (200, 100)


In [9]:
representations = {
    "TF-IDF": (X_train_tfidf, X_test_tfidf),
    "Word2Vec": (X_train_embed, X_test_embed),
}

## Model & Evaluasi

In [10]:
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return acc, prec_macro, rec_macro, f1_macro

In [11]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test, show_report=True):
    """Latih model & tampilkan 4 metrik + classification_report."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc, prec_macro, rec_macro, f1_macro = compute_metrics(y_test, y_pred)
    
    print(f"\n=== {name} ===")
    print("Accuracy       :", acc)
    print("Precision macro:", prec_macro)
    print("Recall macro   :", rec_macro)
    print("F1 macro       :", f1_macro)
    
    if show_report:
        print("\nClassification report:")
        print(classification_report(y_test, y_pred, zero_division=0))
    
    return {
        "Model": name,
        "Accuracy": acc,
        "Precision_macro": prec_macro,
        "Recall_macro": rec_macro,
        "F1_macro": f1_macro
    }


In [12]:
def grid_search_and_evaluate(name, base_model, param_grid,
                             X_train, y_train, X_test, y_test,
                             show_report=True):
    """
    GridSearchCV (scoring F1 macro), lalu evaluasi model terbaik di test set,
    tampilkan 4 metrik + classification_report.
    """
    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=3,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc, prec_macro, rec_macro, f1_macro = compute_metrics(y_test, y_pred)
    
    print(f"\n=== {name} (Tuned) ===")
    print("Best params    :", grid.best_params_)
    print("Accuracy       :", acc)
    print("Precision macro:", prec_macro)
    print("Recall macro   :", rec_macro)
    print("F1 macro       :", f1_macro)
    
    if show_report:
        print("\nClassification report (best model):")
        print(classification_report(y_test, y_pred, zero_division=0))
    
    return {
        "Model": name,
        "Best_Params": grid.best_params_,
        "Accuracy": acc,
        "Precision_macro": prec_macro,
        "Recall_macro": rec_macro,
        "F1_macro": f1_macro
    }


In [13]:
def grid_search_and_evaluate(name, base_model, param_grid,
                             X_train, y_train, X_test, y_test):
    """
    GridSearchCV dengan scoring F1 macro, lalu evaluasi 4 metric di test set.
    Tidak menampilkan classification_report.
    """
    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=3,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=1    # kalau mau hilang tulisan "Fitting 3 folds..." → ganti 0
    )
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    
    print(f"\n=== {name} (Tuned) ===")
    print("Best params    :", grid.best_params_)
    print("Accuracy       :", acc)
    print("Precision macro:", prec_macro)
    print("Recall macro   :", rec_macro)
    print("F1 macro       :", f1_macro)
    
    return {
        "Model": name,
        "Best_Params": grid.best_params_,
        "Accuracy": acc,
        "Precision_macro": prec_macro,
        "Recall_macro": rec_macro,
        "F1_macro": f1_macro
    }


In [14]:
# Dua algoritma ML yang dipakai
base_models = {
    "LogReg": LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        n_jobs=-1
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

# Hyperparameter untuk tuning nanti
param_grids = {
    "LogReg": {
        "C": [0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
        "max_iter": [1000, 3000]
    },
    "RandomForest": {
        "n_estimators": [200, 500],
        "max_depth": [None, 20],
        "max_features": ["sqrt", "log2"]
    }
}


In [15]:
baseline_results = []

for rep_name, (Xtr, Xte) in representations.items():
    for model_name, model in base_models.items():
        name = f"Baseline {model_name} + {rep_name}"
        res = evaluate_model(name, model, Xtr, y_train, Xte, y_test)
        baseline_results.append(res)

baseline_df = pd.DataFrame(baseline_results)
baseline_df



=== Baseline LogReg + TF-IDF ===
Accuracy       : 0.685
Precision macro: 0.2702194357366771
Recall macro   : 0.32463410333274556
F1 macro       : 0.29365079365079366

Classification report:
              precision    recall  f1-score   support

           1       0.70      0.94      0.80       107
           2       0.00      0.00      0.00        16
           3       0.00      0.00      0.00        12
           4       0.00      0.00      0.00        12
           5       0.65      0.68      0.67        53

    accuracy                           0.69       200
   macro avg       0.27      0.32      0.29       200
weighted avg       0.55      0.69      0.61       200


=== Baseline RandomForest + TF-IDF ===
Accuracy       : 0.63
Precision macro: 0.24523087293562523
Recall macro   : 0.3059777816963498
F1 macro       : 0.2721589330369547

Classification report:
              precision    recall  f1-score   support

           1       0.69      0.83      0.75       107
           2    

,Model,Accuracy,Precision_macro,Recall_macro,F1_macro
0,Baseline LogReg + TF-IDF,0.685,0.270219,0.324634,0.293651
1,Baseline RandomForest + TF-IDF,0.630,0.245231,0.305978,0.272159
2,Baseline LogReg + Word2Vec,0.630,0.239048,0.294551,0.263581
3,Baseline RandomForest + Word2Vec,0.650,0.249568,0.305837,0.274538


In [16]:
tuned_results = []

for rep_name, (Xtr, Xte) in representations.items():
    for model_name, base_model in base_models.items():
        name = f"{model_name} + {rep_name}"
        param_grid = param_grids[model_name]
        res = grid_search_and_evaluate(
            name, base_model, param_grid,
            Xtr, y_train, Xte, y_test
        )
        tuned_results.append(res)

tuned_df = pd.DataFrame(tuned_results)
tuned_df


Fitting 3 folds for each of 12 candidates, totalling 36 fits

=== LogReg + TF-IDF (Tuned) ===
Best params    : {'C': 0.5, 'class_weight': 'balanced', 'max_iter': 1000}
Accuracy       : 0.51
Precision macro: 0.3690537182614982
Recall macro   : 0.38516649033092343
F1 macro       : 0.3667800177035859
Fitting 3 folds for each of 8 candidates, totalling 24 fits

=== RandomForest + TF-IDF (Tuned) ===
Best params    : {'max_depth': None, 'max_features': 'log2', 'n_estimators': 200}
Accuracy       : 0.66
Precision macro: 0.25636574074074076
Recall macro   : 0.3171927349673779
F1 macro       : 0.2835205198841563
Fitting 3 folds for each of 12 candidates, totalling 36 fits

=== LogReg + Word2Vec (Tuned) ===
Best params    : {'C': 1.0, 'class_weight': 'balanced', 'max_iter': 1000}
Accuracy       : 0.47
Precision macro: 0.36840167761093506
Recall macro   : 0.3789396343972256
F1 macro       : 0.34678876678876674
Fitting 3 folds for each of 8 candidates, totalling 24 fits

=== RandomForest + Word2Ve

,Model,Best_Params,Accuracy,Precision_macro,Recall_macro,F1_macro
0,LogReg + TF-IDF,"{'C': 0.5, 'class_weight': 'balanced', 'max_it...",0.510,0.369054,0.385166,0.366780
1,RandomForest + TF-IDF,"{'max_depth': None, 'max_features': 'log2', 'n...",0.660,0.256366,0.317193,0.283521
2,LogReg + Word2Vec,"{'C': 1.0, 'class_weight': 'balanced', 'max_it...",0.470,0.368402,0.378940,0.346789
3,RandomForest + Word2Vec,"{'max_depth': 20, 'max_features': 'sqrt', 'n_e...",0.655,0.251283,0.309610,0.277277


### Imbalance Data Handling

In [17]:
print("Distribusi y_train sebelum resampling:")
print(y_train.value_counts().sort_index())


Distribusi y_train sebelum resampling:
rating
1    425
2     65
3     48
4     49
5    212
Name: count, dtype: int64


In [18]:
# TF-IDF → SMOTETomek (butuh dense array)
smote_tomek_tfidf = SMOTETomek(random_state=42)
X_train_tfidf_dense = X_train_tfidf.toarray()

X_train_tfidf_res, y_train_tfidf_res = smote_tomek_tfidf.fit_resample(
    X_train_tfidf_dense, y_train
)

print("\nDistribusi setelah SMOTETomek (TF-IDF):")
print(pd.Series(y_train_tfidf_res).value_counts().sort_index())
print("Shape train TF-IDF resampled:", X_train_tfidf_res.shape)



Distribusi setelah SMOTETomek (TF-IDF):
rating
1    424
2    425
3    425
4    425
5    424
Name: count, dtype: int64
Shape train TF-IDF resampled: (2123, 661)


In [19]:
# Word2Vec embedding → SMOTETomek langsung
smote_tomek_embed = SMOTETomek(random_state=42)

X_train_embed_res, y_train_embed_res = smote_tomek_embed.fit_resample(
    X_train_embed, y_train
)

print("\nDistribusi setelah SMOTETomek (Word2Vec):")
print(pd.Series(y_train_embed_res).value_counts().sort_index())
print("Shape train Embed resampled:", X_train_embed_res.shape)



Distribusi setelah SMOTETomek (Word2Vec):
rating
1    422
2    424
3    425
4    425
5    421
Name: count, dtype: int64
Shape train Embed resampled: (2117, 100)


In [20]:
representations_resampled = {
    "TF-IDF": (X_train_tfidf_res, X_test_tfidf),
    "Word2Vec": (X_train_embed_res, X_test_embed),
}


In [21]:
baseline_smote_results = []

for rep_name, (Xtr_res, Xte) in representations_resampled.items():
    for model_name, model in base_models.items():
        name = f"Baseline {model_name} + {rep_name} (SMOTETomek)"
        res = evaluate_model(name, model, Xtr_res, 
                             y_train_tfidf_res if rep_name=="TF-IDF" else y_train_embed_res,
                             Xte, y_test)
        baseline_smote_results.append(res)

baseline_smote_df = pd.DataFrame(baseline_smote_results)
baseline_smote_df



=== Baseline LogReg + TF-IDF (SMOTETomek) ===
Accuracy       : 0.555
Precision macro: 0.3725021533161068
Recall macro   : 0.3864405160759419
F1 macro       : 0.37344131841880135

Classification report:
              precision    recall  f1-score   support

           1       0.80      0.64      0.72       107
           2       0.04      0.06      0.05        16
           3       0.11      0.17      0.13        12
           4       0.28      0.42      0.33        12
           5       0.63      0.64      0.64        53

    accuracy                           0.56       200
   macro avg       0.37      0.39      0.37       200
weighted avg       0.62      0.56      0.58       200


=== Baseline RandomForest + TF-IDF (SMOTETomek) ===
Accuracy       : 0.585
Precision macro: 0.24575084500241431
Recall macro   : 0.2929642038441192
F1 macro       : 0.26537467700258394

Classification report:
              precision    recall  f1-score   support

           1       0.72      0.73      0.72

,Model,Accuracy,Precision_macro,Recall_macro,F1_macro
0,Baseline LogReg + TF-IDF (SMOTETomek),0.555,0.372502,0.386441,0.373441
1,Baseline RandomForest + TF-IDF (SMOTETomek),0.585,0.245751,0.292964,0.265375
2,Baseline LogReg + Word2Vec (SMOTETomek),0.450,0.361003,0.362737,0.331467
3,Baseline RandomForest + Word2Vec (SMOTETomek),0.610,0.358249,0.359193,0.357724


In [22]:
tuned_smote_results = []

for rep_name, (Xtr_res, Xte) in representations_resampled.items():
    y_res = y_train_tfidf_res if rep_name == "TF-IDF" else y_train_embed_res
    for model_name, base_model in base_models.items():
        name = f"{model_name} + {rep_name} (SMOTETomek)"
        param_grid = param_grids[model_name]
        res = grid_search_and_evaluate(
            name, base_model, param_grid,
            Xtr_res, y_res, Xte, y_test
        )
        tuned_smote_results.append(res)

tuned_smote_df = pd.DataFrame(tuned_smote_results)
tuned_smote_df


Fitting 3 folds for each of 12 candidates, totalling 36 fits

=== LogReg + TF-IDF (SMOTETomek) (Tuned) ===
Best params    : {'C': 2.0, 'class_weight': 'balanced', 'max_iter': 1000}
Accuracy       : 0.575
Precision macro: 0.3658525149190111
Recall macro   : 0.3810240698289543
F1 macro       : 0.37006547848149085
Fitting 3 folds for each of 8 candidates, totalling 24 fits

=== RandomForest + TF-IDF (SMOTETomek) (Tuned) ===
Best params    : {'max_depth': None, 'max_features': 'log2', 'n_estimators': 500}
Accuracy       : 0.61
Precision macro: 0.3127127127127127
Recall macro   : 0.3228207841062717
F1 macro       : 0.2969939294353918
Fitting 3 folds for each of 12 candidates, totalling 36 fits

=== LogReg + Word2Vec (SMOTETomek) (Tuned) ===
Best params    : {'C': 2.0, 'class_weight': None, 'max_iter': 1000}
Accuracy       : 0.445
Precision macro: 0.3653340798805525
Recall macro   : 0.37185608064421327
F1 macro       : 0.3347795943893505
Fitting 3 folds for each of 8 candidates, totalling 24

,Model,Best_Params,Accuracy,Precision_macro,Recall_macro,F1_macro
0,LogReg + TF-IDF (SMOTETomek),"{'C': 2.0, 'class_weight': 'balanced', 'max_it...",0.575,0.365853,0.381024,0.370065
1,RandomForest + TF-IDF (SMOTETomek),"{'max_depth': None, 'max_features': 'log2', 'n...",0.610,0.312713,0.322821,0.296994
2,LogReg + Word2Vec (SMOTETomek),"{'C': 2.0, 'class_weight': None, 'max_iter': 1...",0.445,0.365334,0.371856,0.334780
3,RandomForest + Word2Vec (SMOTETomek),"{'max_depth': 20, 'max_features': 'sqrt', 'n_e...",0.600,0.334973,0.334193,0.333468
